#### **`Dependencies`**

In [2]:
%%bash
cd ..
# rm data/*
# rm -r logs
# rm -r src/__pycache__
tree

.
├── LICENSE
├── README.md
├── config.yaml
├── data
├── notebooks
│   └── ingest_transform_train.ipynb
├── poetry.lock
├── pyproject.toml
└── src
    ├── __init__.py
    ├── config.py
    ├── ingest.py
    ├── logger.py
    ├── train.py
    └── transform.py

4 directories, 12 files


In [3]:
import warnings

import pandas as pd
import polars as pl
import plotly.express as px

from dotenv import load_dotenv
from lightgbm import LGBMRegressor
from plotly.graph_objects import Figure
from xgboost import XGBRegressor

from src.config import Config
from src.ingest import download_file
from src.train import compute_metrics, split_data, train_model
from src.transform import plot_record, tabularize_data

warnings.filterwarnings("ignore")
load_dotenv(Config.HOME_DIR / ".env")

True

In [4]:
# set the Pandas display options
pd.set_option("display.max_rows", 100) # max number of rows to display
pd.set_option("display.max_columns", None) # max number of columns to display

# set the Polars display options
pl.Config(tbl_rows=10) # max number of rows to display
pl.Config(tbl_cols=None) # max number of columns to display
pl.Config(fmt_str_lengths=25) # max number of characters to display for pl.Utf8 (str) dtype columns
pl.Config(fmt_table_cell_list_len=20) # max number of items to display for pl.List dtype columns

#### **`Data ingestion`**

In [5]:
# download, validate, and pre-process raw data from 2022-01
df: pd.DataFrame = download_file(2022, 1)
df

100%|██████████| 257/257 [00:03<00:00, 74.37it/s]


,location_id,pickup_datetime,rides
0,1,2022-01-01 04:00:00,1.0
1,1,2022-01-01 05:00:00,1.0
2,1,2022-01-01 06:00:00,0.0
3,1,2022-01-01 07:00:00,2.0
4,1,2022-01-01 08:00:00,0.0
...,...,...,...
175650,265,2022-01-31 19:00:00,12.0
175651,265,2022-01-31 20:00:00,14.0
175652,265,2022-01-31 21:00:00,1.0
175653,265,2022-01-31 22:00:00,4.0


In [6]:
# confirm that the 'df' pd.DataFrame is free of null values and duplicates
assert df.isna().sum().sum() == 0
assert df.duplicated().sum() == 0

In [7]:
# a list of select location IDs
location_ids: list[int] = [43, 90, 107]

# plot the hourly taxi rides for the selected location IDs
fig: Figure = px.line(
    df.query(f"location_id.isin({location_ids})"),
    x="pickup_datetime",
    y="rides",
    color="location_id",
    labels={
        "pickup_datetime": "Datetime", 
        "rides": "Number of taxi rides", 
        "location_id": "Location ID"
    },
    title="NYC Hourly Taxi Rides",
    template="plotly_dark"
)
fig.show()

#### **`Data transformation`**

In [8]:
# download, validate, pre-process, and transform raw data from 2022-01 into an ML-ready dataset
# NOTE: for the 'day_of_week' feature, 0 == Monday, ..., 6 == Sunday
df: pd.DataFrame = download_file(2022, 1).pipe(tabularize_data)
df

100%|██████████| 257/257 [00:08<00:00, 30.54it/s]


,location_id,pickup_datetime,hour,day_of_week,avg_24_lags,avg_20_lags,avg_16_lags,avg_12_lags,avg_8_lags,avg_4_lags,lag_24,lag_23,lag_22,lag_21,lag_20,lag_19,lag_18,lag_17,lag_16,lag_15,lag_14,lag_13,lag_12,lag_11,lag_10,lag_9,lag_8,lag_7,lag_6,lag_5,lag_4,lag_3,lag_2,lag_1,target
0,1,2022-01-02 04:00:00,4,6,1.708333,1.85,2.1250,1.000000,0.250,0.00,1.0,1.0,0.0,2.0,0.0,0.0,1.0,2.0,1.0,5.0,5.0,11.0,3.0,3.0,1.0,3.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,1,2022-01-02 05:00:00,5,6,1.666667,1.85,2.0625,0.750000,0.000,0.00,1.0,0.0,2.0,0.0,0.0,1.0,2.0,1.0,5.0,5.0,11.0,3.0,3.0,1.0,3.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4.0
2,1,2022-01-02 06:00:00,6,6,1.791667,2.05,2.0000,0.833333,0.500,1.00,0.0,2.0,0.0,0.0,1.0,2.0,1.0,5.0,5.0,11.0,3.0,3.0,1.0,3.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4.0,1.0
3,1,2022-01-02 07:00:00,7,6,1.833333,2.05,1.7500,0.833333,0.625,1.25,2.0,0.0,0.0,1.0,2.0,1.0,5.0,5.0,11.0,3.0,3.0,1.0,3.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4.0,1.0,2.0
4,1,2022-01-02 08:00:00,8,6,1.833333,2.05,1.1875,0.750000,0.875,1.75,0.0,0.0,1.0,2.0,1.0,5.0,5.0,11.0,3.0,3.0,1.0,3.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4.0,1.0,2.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
169597,265,2022-01-31 19:00:00,19,0,12.500000,14.05,16.6250,19.250000,20.000,24.00,7.0,5.0,2.0,5.0,4.0,8.0,3.0,0.0,3.0,6.0,13.0,13.0,24.0,17.0,16.0,14.0,8.0,18.0,20.0,18.0,29.0,21.0,21.0,25.0,12.0
169598,265,2022-01-31 20:00:00,20,0,12.708333,14.45,17.1875,18.250000,20.500,19.75,5.0,2.0,5.0,4.0,8.0,3.0,0.0,3.0,6.0,13.0,13.0,24.0,17.0,16.0,14.0,8.0,18.0,20.0,18.0,29.0,21.0,21.0,25.0,12.0,14.0
169599,265,2022-01-31 21:00:00,21,0,13.083333,14.75,17.6875,18.000000,20.000,18.00,2.0,5.0,4.0,8.0,3.0,0.0,3.0,6.0,13.0,13.0,24.0,17.0,16.0,14.0,8.0,18.0,20.0,18.0,29.0,21.0,21.0,25.0,12.0,14.0,1.0
169600,265,2022-01-31 22:00:00,22,0,13.041667,14.65,16.9375,16.750000,17.625,13.00,5.0,4.0,8.0,3.0,0.0,3.0,6.0,13.0,13.0,24.0,17.0,16.0,14.0,8.0,18.0,20.0,18.0,29.0,21.0,21.0,25.0,12.0,14.0,1.0,4.0


In [9]:
# plot a single record's (row's) lag features (blue) and target (green)
plot_record(df)

#### **`Model training and evaluation`**

In [10]:
# download, validate, pre-process, and transform raw data from 2023-05 into an ML-ready dataset
df: pd.DataFrame = download_file(2023, 5).pipe(tabularize_data)

100%|██████████| 261/261 [00:08<00:00, 30.43it/s]


In [11]:
# split the 'df' pd.DataFrame into train and test sets
Xtrain, ytrain, Xtest, ytest = split_data(df)

In [12]:
# extract the model
model: LGBMRegressor | XGBRegressor = train_model(df)

# fit the model to the train set and evaluate on the test set
if isinstance(model, XGBRegressor):
    model.fit(Xtrain, ytrain, eval_set=[(Xtest, ytest)], verbose=False)
elif isinstance(model, LGBMRegressor):
    model.fit(Xtrain, ytrain, eval_set=[(Xtest, ytest)])

# compute the test set's RMSE and R²
compute_metrics(ytest, model.predict(Xtest))

100%|██████████| 3/3 [00:01<00:00,  2.50it/s]


{'rmse': 9.676, 'r_squared': 0.963}